<h1 align="center">Data Cleaning</h1>

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [21]:
if not os.path.isdir('dataset') and os.path.isdir('../dataset'):
    os.chdir('..')

filename = os.path.join('dataset', '311_Service_Requests_from_2020_to_Present.zip')
df = pd.read_csv(filename, low_memory=False)
df.head()

,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location
0,70471747,09/20/2026 01:51:01 AM,NaN,NYPD,New York City Police Department,Illegal Parking,Blocked Hydrant,NaN,Street/Sidewalk,10016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.744553,-73.977398,POINT (-73.977397686134 40.744552551425)
1,70474903,09/20/2026 01:50:51 AM,NaN,DEP,Department of Environmental Protection,Water Maintenance,Water Main Break,NaN,Street,11238,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.672907,-73.959725,POINT (-73.959725084153 40.672906659623)
2,70470200,09/20/2026 01:50:51 AM,NaN,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11373,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.737271,-73.879943,POINT (-73.879942579006 40.737270748449)
3,70475185,09/20/2026 01:50:45 AM,NaN,NYPD,New York City Police Department,Noise - Vehicle,Car/Truck Music,NaN,Street/Sidewalk,11220,...,Car,NaN,NaN,NaN,NaN,NaN,NaN,40.641947,-73.999272,POINT (-73.999272130506 40.641947092776)
4,70475938,09/20/2026 01:50:40 AM,NaN,NYPD,New York City Police Department,Blocked Driveway,No Access,NaN,Street/Sidewalk,11433,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.689879,-73.791066,POINT (-73.791066478538 40.689878873276)


In [22]:
df['Created Date'] = pd.to_datetime(df['Created Date'], errors='coerce')
df['Closed Date'] = pd.to_datetime(df['Closed Date'], errors='coerce')

missing_closed_date = df['Closed Date'].isna().sum()
print(f"Number of missing 'Closed Date' values: {missing_closed_date}")
df = df.dropna(subset=['Closed Date'])

df['resolution_hours'] = (df['Closed Date'] - df['Created Date']).dt.total_seconds() / 3600

negative_resolution = (df['resolution_hours'] < 0).sum()
print(f"Number of rows with negative resolution time: {negative_resolution}")
df = df[df['resolution_hours'] >= 0]

Number of missing 'Closed Date' values: 179879
Number of rows with negative resolution time: 1011


In [23]:
df.head()

,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location,resolution_hours
75,70475296,2026-09-20 01:39:36,2026-09-20 01:47:32,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11208,...,NaN,NaN,NaN,NaN,NaN,NaN,40.666362,-73.876732,POINT (-73.876732316369 40.666361759266),0.132222
90,70471164,2026-09-20 01:37:27,2026-09-20 01:47:26,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,NaN,Street/Sidewalk,11222,...,NaN,NaN,NaN,NaN,NaN,NaN,40.730319,-73.960665,POINT (-73.960664569783 40.73031925345),0.166389
95,70470378,2026-09-20 01:36:25,2026-09-20 01:45:23,NYPD,New York City Police Department,Noise - Park,Loud Music/Party,NaN,Park/Playground,10033,...,NaN,NaN,NaN,NaN,NaN,NaN,40.843501,-73.945906,POINT (-73.945905791727 40.843500551231),0.149444
99,70478136,2026-09-20 01:35:40,2026-09-20 01:49:29,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Talking,NaN,Street/Sidewalk,11238,...,NaN,NaN,NaN,NaN,NaN,NaN,40.676053,-73.961223,POINT (-73.961222929757 40.676052686722),0.230278
106,70478077,2026-09-20 01:34:30,2026-09-20 01:44:49,NYPD,New York City Police Department,Noise - Commercial,Banging/Pounding,NaN,Club/Bar/Restaurant,10002,...,NaN,NaN,NaN,NaN,NaN,NaN,40.718013,-73.989896,POINT (-73.989895597843 40.718012546099),0.171944


### Select feature columns

In [24]:
selected_columns = [
    'Created Date',
    'Problem (formerly Complaint Type)',
    'Problem Detail (formerly Descriptor)',
    'Agency',
    'Borough',
    'Location Type',
    'Open Data Channel Type',
    'Community Board',
    'Police Precinct',
    'resolution_hours',
]

df = df[selected_columns]
print(f"Shape: {df.shape}")
df.head()

Shape: (3759919, 10)


,Created Date,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Agency,Borough,Location Type,Open Data Channel Type,Community Board,Police Precinct,resolution_hours
75,2026-09-20 01:39:36,Noise - Residential,Loud Music/Party,NYPD,BROOKLYN,Residential Building/House,ONLINE,05 BROOKLYN,Precinct 75,0.132222
90,2026-09-20 01:37:27,Noise - Street/Sidewalk,Loud Music/Party,NYPD,BROOKLYN,Street/Sidewalk,MOBILE,01 BROOKLYN,Precinct 94,0.166389
95,2026-09-20 01:36:25,Noise - Park,Loud Music/Party,NYPD,MANHATTAN,Park/Playground,ONLINE,12 MANHATTAN,Precinct 33,0.149444
99,2026-09-20 01:35:40,Noise - Street/Sidewalk,Loud Talking,NYPD,BROOKLYN,Street/Sidewalk,MOBILE,08 BROOKLYN,Precinct 77,0.230278
106,2026-09-20 01:34:30,Noise - Commercial,Banging/Pounding,NYPD,MANHATTAN,Club/Bar/Restaurant,PHONE,03 MANHATTAN,Precinct 7,0.171944


### Missing values check

In [25]:
missing = df.isna().sum().sort_values(ascending=False)
print (missing)

Location Type                           463792
Problem Detail (formerly Descriptor)     73601
Created Date                                 0
Problem (formerly Complaint Type)            0
Agency                                       0
Borough                                      0
Open Data Channel Type                       0
Community Board                              0
Police Precinct                              0
resolution_hours                             0
dtype: int64


`Location Type` column has 463,792 missing values and `Problem Detail` has 73,601 missing values-> too many to drop -> fill it with `Not Specified`

In [26]:
df['Location Type'] = df['Location Type'].fillna('Not Specified')
df['Problem Detail (formerly Descriptor)'] = df['Problem Detail (formerly Descriptor)'].fillna('Not Specified')
missing = df.isna().sum().sort_values(ascending=False)
print (missing)

Created Date                            0
Problem (formerly Complaint Type)       0
Problem Detail (formerly Descriptor)    0
Agency                                  0
Borough                                 0
Location Type                           0
Open Data Channel Type                  0
Community Board                         0
Police Precinct                         0
resolution_hours                        0
dtype: int64


### Standardize categorical casing

Switched to a simpler approach: lowercase every categorical column entirely, rather than only fixing the ALL CAPS values. This avoids the apostrophe-mangling issue `.title()` has (`"Doctor's Office"` → `"Doctor'S Office"`) and guarantees casing can never cause a false "different category" again, for any column, not just the two checked so far.

In [38]:
categorical_cols = df.select_dtypes(include='object').columns
before_n = {col: df[col].nunique() for col in categorical_cols}

for col in categorical_cols:
    df[col] = df[col].str.lower()

for col in categorical_cols:
    print(f"{col}: {before_n[col]} -> {df[col].nunique()} unique values")

Problem (formerly Complaint Type): 189 -> 189 unique values
Problem Detail (formerly Descriptor): 1006 -> 1006 unique values
Agency: 16 -> 16 unique values
Borough: 6 -> 6 unique values
Location Type: 152 -> 152 unique values
Open Data Channel Type: 5 -> 5 unique values
Community Board: 77 -> 77 unique values
Police Precinct: 78 -> 78 unique values


The number of the unique values is dropped but not significantly -> there is still an issue with the naming

#### Complaint Type

In [41]:
noise = df['Problem (formerly Complaint Type)'].value_counts(dropna=False)
noise[noise.index.astype(str).str.contains('noise', case=False, na=False)]

Problem (formerly Complaint Type)
noise - residential         433900
noise - street/sidewalk     175293
noise - commercial           60070
noise                        58140
noise - vehicle              51040
noise - park                  8460
noise - house of worship      2920
noise - helicopter             181
Name: count, dtype: int64

In [45]:
col = 'Problem (formerly Complaint Type)'
noise_df = df[df[col].str.contains('noise', case=False, na=False)]
noise_summary = noise_df.groupby(col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
noise_summary

,count,median_hours
Problem (formerly Complaint Type),,
noise - general,731683,1.120278
noise,58140,68.950000
noise - helicopter,181,6550.935278


In [42]:
def merge_categories(df, column, values_to_merge, new_value):
    df[column] = df[column].replace({v: new_value for v in values_to_merge})
    return df

In [43]:
noise_to_merge = [
    'noise - house of worship',
    'noise - commercial',
    'noise - vehicle',
    'noise - park',
    'noise - street/sidewalk',
    'noise - residential',
]

df = merge_categories(df, 'Problem (formerly Complaint Type)', noise_to_merge, 'noise - general')

Same check, scanned for other complaint types that share a `"prefix - subtype"` pattern like noise did. Two more genuine families turned up: `street sign` and `highway sign` — both homogeneous enough (no extreme outlier like helicopter) to merge without excluding anything.

In [46]:
sign = df['Problem (formerly Complaint Type)'].value_counts(dropna=False)
sign[sign.index.astype(str).str.contains('sign', case=False, na=False)]

Problem (formerly Complaint Type)
traffic signal condition    56922
street sign - damaged        5082
street sign - dangling       3870
street sign - missing        3076
highway sign - damaged         73
highway sign - missing         39
highway sign - dangling        19
Name: count, dtype: int64

In [49]:
col = 'Problem (formerly Complaint Type)'
sign_df = df[df[col].str.contains('sign', case=False, na=False)]
sign_summary = sign_df.groupby(col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
sign_summary

,count,median_hours
Problem (formerly Complaint Type),,
traffic signal condition,56922,2.350000
street sign - missing,3076,100.046528
street sign - damaged,5082,111.516250
highway sign - dangling,19,133.852778
street sign - dangling,3870,141.794583
highway sign - damaged,73,145.088889
highway sign - missing,39,227.011667


In [50]:
df = merge_categories(
    df, col,
    ['street sign - missing', 'street sign - damaged', 'street sign - dangling'],
    'street sign',
)

df = merge_categories(
    df, col,
    ['highway sign - missing', 'highway sign - damaged', 'highway sign - dangling'],
    'highway sign',
)

In [53]:
marshal = df['Problem (formerly Complaint Type)'].value_counts(dropna=False)
marshal[marshal.index.astype(str).str.contains('marshal', case=False, na=False)]

Problem (formerly Complaint Type)
building marshal's office    19
Name: count, dtype: int64

In [54]:
sub = df[df[col].isin(["building marshal's office", 'building marshals office'])]
summary = sub.groupby(col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)

                                   count  median_hours
Problem (formerly Complaint Type)                     
building marshal's office             19    299.636944


In [55]:
df = merge_categories(df, col, ["building marshal's office", 'building marshals office'], "building marshal's office")

In [56]:
hazardous = df['Problem (formerly Complaint Type)'].value_counts(dropna=False)
hazardous[hazardous.index.astype(str).str.contains('hazardous', case=False, na=False)]

Problem (formerly Complaint Type)
hazardous materials    1743
hazardous material      424
Name: count, dtype: int64

In [57]:
sub = df[df[col].isin(['hazardous material', 'hazardous materials'])]
summary = sub.groupby(col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)

                                   count  median_hours
Problem (formerly Complaint Type)                     
hazardous materials                 1743      2.500000
hazardous material                   424      2.520556


In [60]:
df = merge_categories(df, col, ['hazardous material', 'hazardous materials'], 'hazardous materials')

In [61]:
electric = df['Problem (formerly Complaint Type)'].value_counts(dropna=False)
electric[electric.index.astype(str).str.contains('electric', case=False, na=False)]

Problem (formerly Complaint Type)
electric      32986
electrical     2481
Name: count, dtype: int64

In [62]:
sub = df[df[col].isin(['electric', 'electrical'])]
summary = sub.groupby(col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)

Empty DataFrame
Columns: [count, median_hours]
Index: []


In [63]:
df = merge_categories(df, col, ['electric', 'electrical'], 'electrical')

In [64]:
for col in categorical_cols:
    print(f"{col}: {before_n[col]} -> {df[col].nunique()} unique values")

Problem (formerly Complaint Type): 189 -> 178 unique values
Problem Detail (formerly Descriptor): 1006 -> 1006 unique values
Agency: 16 -> 16 unique values
Borough: 6 -> 6 unique values
Location Type: 152 -> 152 unique values
Open Data Channel Type: 5 -> 5 unique values
Community Board: 77 -> 77 unique values
Police Precinct: 78 -> 78 unique values


#### Location Type

## Plan

7. Consolidate rare categories into "Other" (`Problem (formerly Complaint Type)`, `Location Type`)
8. Decide how to handle `Borough == "Unspecified"`
9. Resolve `Due Date` missingness and its role (feature vs. SLA benchmark)
10. Handle remaining missing values, column by column
11. Final check + save a cleaned checkpoint (parquet/csv)